# Cohort LTV roll-forward: validate before forecasting

A mobile-game assessment asked for weekly in-app purchase (IAP) cohorts, cumulative revenue through day 360, and a day-360 estimate for an immature cohort. Its supplied coefficient table used `tier`, while the analysis expected `tier_inapp`; the join therefore stopped before producing forecasts. This notebook demonstrates a reproducible, synthetic alternative: normalize join keys, estimate day-30-to-day-360 factors on older cohorts, and test them on later cohorts whose mature outcomes are hidden during prediction.

**All generated data and results below are synthetic. They are not Belka Games data or assessment findings.**

## Method and limits

- Cohort age is the integer number of days from install to revenue event. Weeks start Monday.
- The target is cumulative IAP revenue at day 360—not profit, ROAS, or total player value.
- For each segment, the factor is the median of mature training-cohort `D360 / D30` ratios. Segments are campaign type and geo tier.
- The last six cohorts are held out. Forecasts use only their D30 revenue; score them against their hidden D360 totals.
- This is an illustrative baseline. A production forecast needs rolling cutoffs, enough mature cohorts per segment, drift checks, uncertainty, and a consistent revenue definition.

In [ ]:
import math
import random
import statistics
from collections import defaultdict

random.seed(41)
CAMPAIGNS = ("prospecting", "retargeting")
TIERS = (1, 2)
N_COHORTS = 30
MAX_DAY = 360
OBSERVATION_DAY = 30
HOLDOUT_N = 6

# Different segments have different revenue scales and maturation speeds.
segment_scale = {("prospecting", 1): 1.0, ("prospecting", 2): 1.45,
                 ("retargeting", 1): 0.78, ("retargeting", 2): 1.12}
segment_tau = {("prospecting", 1): 48, ("prospecting", 2): 64,
               ("retargeting", 1): 34, ("retargeting", 2): 43}

# Synthetic cumulative-revenue curves, with modest cohort-to-cohort variation.
curves = {}
for cohort in range(N_COHORTS):
    for campaign in CAMPAIGNS:
        for tier in TIERS:
            seg = (campaign, tier)
            size_noise = random.uniform(0.86, 1.14)
            shape_noise = random.uniform(0.92, 1.08)
            final = 1000 * segment_scale[seg] * size_noise
            tau = segment_tau[seg] * shape_noise
            curves[(cohort, campaign, tier)] = [
                final * (1 - math.exp(-day / tau)) / (1 - math.exp(-MAX_DAY / tau))
                for day in range(MAX_DAY + 1)
            ]

print(f"Generated {len(curves)} synthetic cohort-segment curves; target horizon D{MAX_DAY}.")

In [ ]:
# Split chronologically. Only older cohorts contribute coefficient estimates.
train_ids = range(N_COHORTS - HOLDOUT_N)
test_ids = range(N_COHORTS - HOLDOUT_N, N_COHORTS)

ratios_by_segment = defaultdict(list)
ratios_global = []
for cohort in train_ids:
    for campaign in CAMPAIGNS:
        for tier in TIERS:
            curve = curves[(cohort, campaign, tier)]
            ratio = curve[MAX_DAY] / curve[OBSERVATION_DAY]
            ratios_by_segment[(campaign, tier)].append(ratio)
            ratios_global.append(ratio)

factors = {segment: statistics.median(values)
           for segment, values in ratios_by_segment.items()}
global_factor = statistics.median(ratios_global)

# Evaluate on later cohorts. D360 is used only after forecast creation for scoring.
rows = []
for cohort in test_ids:
    for campaign in CAMPAIGNS:
        for tier in TIERS:
            curve = curves[(cohort, campaign, tier)]
            observed_d30 = curve[OBSERVATION_DAY]
            actual_d360 = curve[MAX_DAY]
            seg_factor = factors[(campaign, tier)]
            rows.append({
                "cohort": cohort, "campaign_type": campaign, "tier": tier,
                "d30_observed": observed_d30, "actual_d360": actual_d360,
                "segment_forecast_d360": observed_d30 * seg_factor,
                "global_forecast_d360": observed_d30 * global_factor,
            })

def wape(prediction_key):
    numerator = sum(abs(row[prediction_key] - row["actual_d360"]) for row in rows)
    denominator = sum(row["actual_d360"] for row in rows)
    return numerator / denominator

def signed_bias(prediction_key):
    return sum(row[prediction_key] - row["actual_d360"] for row in rows) / sum(row["actual_d360"] for row in rows)

print(f"Training cohorts: {len(list(train_ids))}; holdout cohorts: {len(list(test_ids))}")
print(f"Segment-factor WAPE: {wape('segment_forecast_d360'):.1%}")
print(f"Global-factor WAPE:  {wape('global_forecast_d360'):.1%}")
print(f"Segment-factor signed bias: {signed_bias('segment_forecast_d360'):+.1%}")
print(f"Global-factor signed bias:  {signed_bias('global_forecast_d360'):+.1%}")

In [ ]:
# Inspect sample predictions after the blind forecasts have been scored.
for row in rows[:6]:
    print({key: round(value, 2) if isinstance(value, float) else value
           for key, value in row.items()})

## Data-contract check: normalize names, do not silently drop forecasts

The source task exposed the key mismatch: the revenue table used `tier_inapp`, while the coefficient table used `tier`. Rename the coefficient key explicitly before the join. Then check unmatched rows and coefficient validity. A missing factor should be reported as missing—not replaced with an unreviewed default.

In [ ]:
# Tiny synthetic schema example, matching the mismatch found in the assessment notebook.
revenue_keys = {"app_code", "platform", "campaign_type", "tier_inapp", "cohort_day"}
coefficient_keys = {"app_code", "platform", "campaign_type", "tier", "valid_date", "cohort_day", "coef"}
assert "tier_inapp" in revenue_keys and "tier" in coefficient_keys
normalized_coefficient_keys = {"tier_inapp" if key == "tier" else key for key in coefficient_keys}
assert "tier_inapp" in normalized_coefficient_keys
print("Schema check passed after explicit tier -> tier_inapp normalization.")
print("Production checks: uniqueness of (valid_date, segment, cohort_day), positive factors, join coverage, and minimum mature-cohort count.")

## Interpretation

The synthetic segment multiplier can be compared with a global multiplier using the same held-out cohorts and D30 cutoff. The numbers are a property of this generated example, not a performance claim about any game or production model. If a segment has too few mature training cohorts, withhold its prediction or use a documented fallback with an uncertainty flag.

The assessment's single-cohort power curve is also not enough to establish D360 reliability. A power function can produce a negative value at day zero and continue growing rather than approaching a limit. Compare candidate curves and multiplier baselines in chronological backtests before choosing one. Freeze each forecast at its historical cutoff, then score it only when the cohort matures.

This data contains IAP revenue only. It does not support conclusions about ad revenue, acquisition profitability, ROAS, or causal channel performance.